In [ ]:
"""
Diabetic Retinopathy Screening Pipeline
=======================================

End-to-end training and evaluation pipeline for binary referral classification
on the Kaggle Diabetic Retinopathy dataset.

The pipeline uses a strict patient-disjoint three-way split:

    train  -- learn model weights
    cal    -- fit temperature scalers, choose thresholds, tune the veto rule,
              and train the stacking meta-learner
    test   -- final reporting only; never used for tuning

Stage 1 of the referral system is a high-sensitivity ResNet50 classifier.
Stage 2 reviews every stage-1 positive with two additional models
(ViT-B/16 and a ResNet18-based encoder) and applies an agreement-based veto
to remove low-confidence referrals while preserving the sensitivity target.

The pipeline also produces a per-case reversal audit explaining why stage-1
positives were overturned in stage 2, and Grad-CAM visualisations for
representative positive, negative, and reversed cases.

Usage
-----
    python dr_pipeline.py

Most configuration lives in the constants near the top of the file. The
Colab-specific code paths (Google Drive mount, file download) are skipped
gracefully when run outside Colab.
"""

from __future__ import annotations

# Standard library
import json
import math
import os
import random
import shutil
import warnings
import zipfile
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# Third-party
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageOps
from tqdm.auto import tqdm

# Torch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models
from torchvision.transforms import InterpolationMode

# scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    auc,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold

import joblib

warnings.filterwarnings("ignore")


# ============================================================================
# Configuration
# ============================================================================

PIPELINE_VERSION = "3.1.0"
SEED = 1337

# --- paths ---
# The pipeline expects an unzipped dataset at IMAGES_DIR with a labels CSV
# at LABELS_CSV (columns: image, level). If the zip is present and the
# unzipped dir is empty, we extract it. Otherwise we fall back to a tiny
# synthetic dataset so the pipeline can still run end-to-end as a smoke test.

#ZIP_PATH = "/content/drive/MyDrive/train.zip"
#BASE_UNZIP_DIR = "/content/train_unzipped"
#IMAGES_DIR_CONFIG = "/content/train_unzipped/train"
#LABELS_CSV = "/content/trainLabels.csv"
#ROOT_OUT = "/content/dr_pipeline_v3_1_patient_3way"

ZIP_PATH = "/content/drive/MyDrive/train_subset.zip"
BASE_UNZIP_DIR = "/content/train_subset_unzipped"
IMAGES_DIR_CONFIG = "/content/train_subset_unzipped/train"
LABELS_CSV = "/content/trainLabels.csv"
ROOT_OUT = "/content/dr_pipeline_v3_1_patient_3way"

# --- image / training ---
IMAGE_SIZE = 224
NUM_CLASSES = 5
BATCH_SIZE_TRAIN = 16
BATCH_SIZE_EVAL = 64
USE_MIXED_PRECISION = True
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 4

# Cap training images per class to keep wall-clock time manageable when
# the full Kaggle dataset is used. Set to None to disable.
MAX_TRAIN_PER_CLASS = 1500

# Per-model training schedules. Each tuple is (epochs, learning_rate).
RN_EPOCHS, RN_LR = 14, 2e-4
VIT_EPOCHS, VIT_LR = 14, 5e-5
ENC_EPOCHS, ENC_LR = 12, 2e-4

# --- patient-level split fractions ---
TEST_FRAC = 0.15            # of all patients
CAL_FRAC_OF_REMAIN = 0.20   # of the non-test patients

# --- stage 1 / stage 2 ---
STAGE1_MODEL_NAME = "resnet50_finetune"
STAGE1_SENS_TARGET = 0.95
CONSENSUS_MODELS = ["resnet50_finetune", "vit_b16_finetune", "encoder_supervised"]
VETO_MIN_AGREE = 2
VETO_THR_GRID = np.round(np.linspace(0.05, 0.70, 66), 4)

# Two operating modes are tuned on CAL and reported on TEST.
OPERATING_MODES = {
    "high_safety": {"target_sens": 0.95, "label": "High-Safety Mode (sens >= 0.95)"},
    "balanced":    {"target_sens": 0.90, "label": "Balanced Mode (sens >= 0.90)"},
}

# --- stacking ---
STACKING_CV_FOLDS = 5

# --- economics ---
COST_PER_REFERRAL = 250.0
SCREENINGS_BASE = 1000

# --- output sub-directories ---
FILES_DIR = str(Path(ROOT_OUT) / "public")
MODELS_DIR = str(Path(ROOT_OUT) / "models")
CAL_DIR = Path(ROOT_OUT) / "calibration_diagnostics"
TEST_DIR = Path(ROOT_OUT) / "final_test_results"
AUDIT_DIR = Path(ROOT_OUT) / "second_stage_audit"
AUDIT_TEXT_DIR = AUDIT_DIR / "text_reports"
AUDIT_PLOTS_DIR = AUDIT_DIR / "plots"
EXPLAIN_DIR = Path(ROOT_OUT) / "explainability_gradcam"
POSTER_DIR = Path(ROOT_OUT) / "poster_gradcam_panels"

# ImageNet normalisation (used for both training transforms and Grad-CAM)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


# ============================================================================
# Reproducibility
# ============================================================================

def set_seed(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch (CPU + CUDA) for deterministic runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


# ============================================================================
# Data preparation
# ============================================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}


def try_mount_colab_drive() -> None:
    """Mount Google Drive if running inside Colab. Silent no-op otherwise."""
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception:
        pass


def safe_unzip(zip_path: str, out_dir: str) -> bool:
    """Extract a zip file if it exists and is valid. Returns True on success."""
    if not zip_path or not os.path.exists(zip_path) or os.path.isdir(zip_path):
        return False
    if not zipfile.is_zipfile(zip_path):
        print(f"[unzip] Not a valid zip: {zip_path}")
        return False
    try:
        os.makedirs(out_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(out_dir)
        print(f"[unzip] Extracted {zip_path} -> {out_dir}")
        return True
    except Exception as exc:
        print(f"[unzip] Failed: {exc}")
        return False


def list_images(root: str) -> List[str]:
    root_p = Path(root)
    if not root_p.exists():
        return []
    return [str(p) for p in root_p.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]


def directory_has_images(d: Optional[str]) -> bool:
    if not d:
        return False
    p = Path(d)
    if not p.exists():
        return False
    return any(q.suffix.lower() in IMAGE_EXTENSIONS for q in p.rglob("*"))


def find_image_directory(base_dir: str, min_images: int = 1) -> Optional[str]:
    """Look one or two levels deep for a folder that contains image files."""
    base = Path(base_dir)
    if not base.exists():
        return None

    # case 1: images directly under base
    if any(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in base.iterdir()):
        return str(base)

    # case 2: images one level down
    for sub in base.iterdir():
        if sub.is_dir() and any(
            q.is_file() and q.suffix.lower() in IMAGE_EXTENSIONS for q in sub.iterdir()
        ):
            return str(sub)

    # case 3: anywhere under base
    matches = [p for p in base.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
    if len(matches) >= min_images:
        return str(matches[0].parent)
    return None


def generate_synthetic_dataset(root_dir: str, labels_csv: str,
                               n_per_class: int = 40, size: int = 300) -> str:
    """Create a small synthetic dataset so the pipeline can be smoke-tested.

    Each image is a dark RGB canvas with a fake optic disc and a number of
    red blobs scaled by class. Class 0 has few blobs; class 4 has many.
    """
    rng = random.Random(SEED)
    out_root = Path(root_dir) / "train_subset"
    out_root.mkdir(parents=True, exist_ok=True)

    rows: List[Dict[str, Any]] = []
    for cls in range(5):
        for i in range(n_per_class):
            name = f"{cls}_{i:03d}"
            img = Image.new(
                "RGB", (size, size),
                (rng.randint(10, 30), rng.randint(10, 30), rng.randint(10, 30)),
            )
            draw = ImageDraw.Draw(img)

            # fake optic disc
            r = rng.randint(16, 28)
            cx = rng.randint(r + 10, size - r - 10)
            cy = rng.randint(r + 10, size - r - 10)
            draw.ellipse([cx - r, cy - r, cx + r, cy + r],
                         outline=(250, 240, 220), width=3)

            # class-correlated red blobs (microaneurysm/haemorrhage stand-in)
            n_blobs = cls * 8 + rng.randint(0, 5)
            for _ in range(n_blobs):
                x, y = rng.randint(8, size - 8), rng.randint(8, size - 8)
                rad = rng.randint(2, 5)
                colour = (rng.randint(180, 255), rng.randint(50, 120), rng.randint(50, 120))
                draw.ellipse([x - rad, y - rad, x + rad, y + rad], fill=colour)

            img.save(out_root / f"{name}.jpg", quality=92)
            rows.append({"image": name, "level": cls})

    pd.DataFrame(rows).to_csv(labels_csv, index=False)
    print(f"[synth] Wrote {len(rows)} synthetic images to {out_root}")
    return str(out_root)


def crop_black_borders(img: Image.Image, threshold: int = 8) -> Image.Image:
    """Crop near-black borders that are common on fundus photos."""
    grey = img.convert("L")
    bw = grey.point(lambda x: 255 if x > threshold else 0, mode="1")
    bbox = bw.getbbox()
    return img.crop(bbox) if bbox else img


def preprocess_image_to_disk(src_path: str, dst_path: str, size: int) -> bool:
    """Apply the standard preprocessing pipeline and save the result."""
    if Path(dst_path).exists():
        return True
    try:
        with Image.open(src_path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            im = crop_black_borders(im)
            im = ImageOps.autocontrast(im, cutoff=1)
            im = ImageOps.equalize(im)

            # centre square crop, then resize
            w, h = im.size
            s = min(w, h)
            im = im.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))
            im = im.resize((size, size))
            im.save(dst_path, quality=95)
        return True
    except Exception as exc:
        print(f"[preprocess] Skipping {src_path}: {exc}")
        return False


def preprocess_all_images(df_in: pd.DataFrame, out_dir: str, size: int) -> pd.DataFrame:
    """Preprocess every row's `filename` to `out_dir`. Returns a new dataframe
    keyed by the processed path."""
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    out_paths: List[str] = []
    kept_levels: List[int] = []

    print(f"[preprocess] Resizing {len(df_in)} images to {size}x{size}")
    iterator = zip(df_in["filename"], df_in["level"])
    for src, lvl in tqdm(iterator, total=len(df_in)):
        dst = str(Path(out_dir) / Path(src).name)
        if preprocess_image_to_disk(src, dst, size):
            out_paths.append(dst)
            kept_levels.append(int(lvl))

    return pd.DataFrame({"proc_path": out_paths, "level": kept_levels})


def extract_patient_id(filepath: str) -> str:
    """Patient id is the first token of the filename, separated by '_'.

    For the Kaggle DR dataset, files are named e.g. `1234_left.jpeg` and
    `1234_right.jpeg`, so both eyes share patient id `1234`.
    """
    stem = Path(filepath).stem
    parts = stem.split("_")
    return parts[0] if parts else stem


def make_patient_disjoint_split(
    df: pd.DataFrame,
    test_frac: float,
    cal_frac_of_remain: float,
    seed: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Two-stage GroupShuffleSplit that holds out test patients first, then
    splits the rest into train / cal. Patient ids never appear in more than
    one of the three subsets.
    """
    gss_test = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=seed)
    idx_rem, idx_test = next(
        gss_test.split(df["proc_path"], df["level"], groups=df["patient_id"])
    )
    df_remain = df.iloc[idx_rem].reset_index(drop=True)
    df_test = df.iloc[idx_test].reset_index(drop=True)

    gss_cal = GroupShuffleSplit(n_splits=1, test_size=cal_frac_of_remain, random_state=seed)
    idx_tr, idx_cal = next(
        gss_cal.split(df_remain["proc_path"], df_remain["level"], groups=df_remain["patient_id"])
    )
    df_train = df_remain.iloc[idx_tr].reset_index(drop=True)
    df_cal = df_remain.iloc[idx_cal].reset_index(drop=True)

    # Sanity check: no patient overlap.
    train_pat = set(df_train["patient_id"])
    cal_pat = set(df_cal["patient_id"])
    test_pat = set(df_test["patient_id"])

    overlaps = {
        "train ∩ cal":  train_pat & cal_pat,
        "train ∩ test": train_pat & test_pat,
        "cal ∩ test":   cal_pat & test_pat,
    }
    bad = {k: len(v) for k, v in overlaps.items() if v}
    if bad:
        raise ValueError(f"Patient overlap detected: {bad}")

    return df_train, df_cal, df_test


def stratified_cap_per_class(df: pd.DataFrame, label_col: str = "level",
                             cap: Optional[int] = MAX_TRAIN_PER_CLASS,
                             seed: int = SEED) -> pd.DataFrame:
    """Down-sample each class to at most `cap` rows."""
    if cap is None or cap <= 0:
        return df.copy().reset_index(drop=True)

    parts = [
        g.sample(n=min(len(g), cap), random_state=seed)
        for _, g in df.groupby(label_col)
    ]
    return pd.concat(parts, ignore_index=True)


# ============================================================================
# Dataset and DataLoaders
# ============================================================================

class FundusDataset(Dataset):
    """Reads pre-processed fundus images from disk and returns (tensor, label)."""

    def __init__(self, df: pd.DataFrame, augment: bool = False):
        self.df = df.reset_index(drop=True)
        if augment:
            self.transform = T.Compose([
                T.Resize(int(IMAGE_SIZE * 1.10), interpolation=InterpolationMode.BILINEAR),
                T.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
                T.RandomHorizontalFlip(p=0.5),
                T.RandomVerticalFlip(p=0.2),
                T.RandomRotation(degrees=10, interpolation=InterpolationMode.BILINEAR, fill=0),
                T.ColorJitter(brightness=0.10, contrast=0.25, saturation=0.10, hue=0.0),
                T.RandomPerspective(distortion_scale=0.15, p=0.25,
                                    interpolation=InterpolationMode.BILINEAR),
                T.ToTensor(),
                T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ])
        else:
            self.transform = T.Compose([
                T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR),
                T.ToTensor(),
                T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ])

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        with Image.open(row.proc_path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            x = self.transform(im)
        return x, int(row.level)


def build_weighted_sampler(df: pd.DataFrame, num_classes: int) -> WeightedRandomSampler:
    """Inverse-frequency sampling so rare classes are over-represented."""
    counts = Counter(df["level"].tolist())
    class_counts = np.array([counts.get(i, 1) for i in range(num_classes)], dtype=np.float32)
    class_weights = class_counts.sum() / (class_counts + 1e-6)
    sample_weights = df["level"].map(
        {i: float(class_weights[i]) for i in range(num_classes)}
    ).values
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights),
                                 replacement=True)


def build_loaders(df_train: pd.DataFrame, df_cal: pd.DataFrame, df_test: pd.DataFrame,
                  device: torch.device) -> Tuple[DataLoader, DataLoader, DataLoader]:
    pin = device.type == "cuda"
    sampler = build_weighted_sampler(df_train, NUM_CLASSES)

    train_loader = DataLoader(
        FundusDataset(df_train, augment=True),
        batch_size=BATCH_SIZE_TRAIN, sampler=sampler,
        num_workers=0, pin_memory=pin,
    )
    cal_loader = DataLoader(
        FundusDataset(df_cal, augment=False),
        batch_size=BATCH_SIZE_EVAL, shuffle=False,
        num_workers=0, pin_memory=pin,
    )
    test_loader = DataLoader(
        FundusDataset(df_test, augment=False),
        batch_size=BATCH_SIZE_EVAL, shuffle=False,
        num_workers=0, pin_memory=pin,
    )
    return train_loader, cal_loader, test_loader


# ============================================================================
# Models
# ============================================================================

def build_resnet50(num_classes: int = NUM_CLASSES) -> nn.Module:
    try:
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    except Exception:
        model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_vit_b16(num_classes: int = NUM_CLASSES) -> nn.Module:
    try:
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    except Exception:
        model = models.vit_b_16(weights=None)

    if hasattr(model, "heads") and hasattr(model.heads, "head"):
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    else:
        model.heads = nn.Linear(model.heads.in_features, num_classes)
    return model


class SupervisedEncoder(nn.Module):
    """A small ResNet18-based classifier used as the third ensemble member."""

    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        try:
            backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        except Exception:
            backbone = models.resnet18(weights=None)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.encoder = backbone
        self.head = nn.Linear(in_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(x))


# ============================================================================
# Training
# ============================================================================

def class_weight_tensor(df: pd.DataFrame, num_classes: int,
                        device: torch.device) -> torch.Tensor:
    """Inverse-frequency class weights, normalised to mean 1."""
    counts = Counter(df["level"].tolist())
    w = np.zeros(num_classes, dtype=np.float32)
    for k in range(num_classes):
        w[k] = 1.0 / max(1, counts.get(k, 1))
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)


@torch.no_grad()
def collect_predictions(model: nn.Module, loader: DataLoader,
                        device: torch.device) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Run `model` over `loader` and return (labels, logits, softmax_probs)."""
    model.eval()
    ys: List[np.ndarray] = []
    logits_chunks: List[np.ndarray] = []
    probs_chunks: List[np.ndarray] = []

    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb)
        probs = torch.softmax(logits, dim=1)
        ys.append(np.asarray(yb))
        logits_chunks.append(logits.detach().cpu().numpy())
        probs_chunks.append(probs.detach().cpu().numpy())

    return (
        np.concatenate(ys),
        np.concatenate(logits_chunks, axis=0),
        np.concatenate(probs_chunks, axis=0),
    )


def to_binary(y_5class) -> np.ndarray:
    """Map 5-class DR labels to binary referrable (>= 1)."""
    return (np.asarray(y_5class, dtype=int) >= 1).astype(int)


def referral_score_from_probs(probs_5class: np.ndarray) -> np.ndarray:
    """P(referrable) = 1 - P(class 0)."""
    return 1.0 - probs_5class[:, 0]


def safe_binary_auc(y_true_bin: np.ndarray, scores: np.ndarray) -> float:
    y_true_bin = np.asarray(y_true_bin).astype(int)
    if len(np.unique(y_true_bin)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true_bin, scores))


def train_classifier(
    model: nn.Module,
    name: str,
    epochs: int,
    lr: float,
    train_loader: DataLoader,
    cal_loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> Tuple[nn.Module, Dict[str, Any]]:
    """Standard fine-tune loop with cosine LR, AMP, and early stopping on
    calibration AUC.
    """
    model = model.to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=max(1, epochs))
    use_amp = USE_MIXED_PRECISION and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best = {"cal_auc": -1.0, "state": None, "epoch": 0}
    epochs_since_improvement = 0

    for ep in range(1, epochs + 1):
        model.train()
        loss_total = 0.0
        for xb, yb in tqdm(train_loader, desc=f"{name} ep {ep}/{epochs}", leave=False):
            xb = xb.to(device)
            yb = torch.as_tensor(yb, device=device, dtype=torch.long)
            optim.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                logits = model(xb)
                loss = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()
            loss_total += float(loss.item())
        sched.step()

        y5_cal, _, probs5_cal = collect_predictions(model, cal_loader, device)
        ybin_cal = to_binary(y5_cal)
        p_ref_cal = referral_score_from_probs(probs5_cal)
        cal_auc = safe_binary_auc(ybin_cal, p_ref_cal)
        train_loss = loss_total / max(1, len(train_loader))
        print(f"[{name}] ep {ep}: cal_auc={cal_auc:.4f}  train_loss={train_loss:.4f}")

        if cal_auc > best["cal_auc"]:
            best = {
                "cal_auc": float(cal_auc),
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "epoch": ep,
            }
            epochs_since_improvement = 0
        else:
            epochs_since_improvement += 1
            if epochs_since_improvement >= EARLY_STOP_PATIENCE:
                print(f"[{name}] early stop at epoch {ep}")
                break

    if best["state"] is not None:
        model.load_state_dict({k: v.to(device) for k, v in best["state"].items()})

    print(f"[{name}] best cal_auc={best['cal_auc']:.4f} at epoch {best['epoch']}")
    return model, best


# ============================================================================
# Threshold selection
# ============================================================================

def threshold_for_min_sensitivity(y_true_bin: np.ndarray, scores: np.ndarray,
                                  target_sens: float = 0.90) -> Dict[str, Any]:
    """Pick the threshold from the ROC curve that meets `target_sens` with the
    lowest false-positive rate. Falls back gracefully on degenerate inputs.
    """
    y_true_bin = np.asarray(y_true_bin).astype(int)
    scores = np.asarray(scores).astype(float)

    if len(np.unique(y_true_bin)) < 2:
        thr = float(np.min(scores)) if len(scores) else 0.0
        y_pred = (scores >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred, labels=[0, 1]).ravel()
        return {
            "thr": thr,
            "sens": float(tp / max(1, tp + fn)),
            "spec": float(tn / max(1, tn + fp)),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        }

    fpr, tpr, thresholds = roc_curve(y_true_bin, scores)
    thresholds = np.array(thresholds)
    # roc_curve sometimes prepends np.inf
    inf_mask = np.isinf(thresholds)
    if inf_mask.any():
        thresholds[inf_mask] = np.max(scores) + 1e-6

    valid = tpr >= target_sens
    if not valid.any():
        idx = len(thresholds) - 1
    else:
        valid_idxs = np.where(valid)[0]
        idx = int(valid_idxs[np.argmin(fpr[valid_idxs])])
        idx = min(idx, len(thresholds) - 1)

    thr = float(thresholds[idx])
    y_pred = (scores >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred, labels=[0, 1]).ravel()
    return {
        "thr": thr,
        "sens": float(tp / max(1, tp + fn)),
        "spec": float(tn / max(1, tn + fp)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def sensitivity_is_met(y_true_bin: np.ndarray, y_pred: np.ndarray,
                       target_sens: float, tol: float = 1e-3) -> Tuple[bool, float]:
    _, _, fn, tp = confusion_matrix(y_true_bin, y_pred, labels=[0, 1]).ravel()
    actual = tp / max(1, tp + fn)
    return actual >= (target_sens - tol), float(actual)


# ============================================================================
# Calibration
# ============================================================================

class TemperatureScaler(nn.Module):
    """A single learnable temperature parameter applied to pre-softmax logits."""

    def __init__(self):
        super().__init__()
        self.log_T = nn.Parameter(torch.zeros((), dtype=torch.float32))

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        T = torch.exp(self.log_T).clamp(0.5, 10.0)
        return logits / T


def fit_temperature(logits_np: np.ndarray, y_np: np.ndarray,
                    device: torch.device, max_iter: int = 50) -> float:
    """Fit a temperature scalar by minimising NLL with LBFGS on CAL."""
    logits = torch.tensor(logits_np, dtype=torch.float32, device=device)
    y = torch.tensor(y_np, dtype=torch.long, device=device)
    ts = TemperatureScaler().to(device)
    nll = nn.CrossEntropyLoss()
    optim = torch.optim.LBFGS(ts.parameters(), lr=0.25, max_iter=max_iter)

    def closure():
        optim.zero_grad(set_to_none=True)
        loss = nll(ts(logits), y)
        loss.backward()
        return loss

    optim.step(closure)
    return float(torch.exp(ts.log_T).detach().cpu().item())


def apply_temperature(logits_np: np.ndarray, T: float) -> np.ndarray:
    T = float(np.clip(T, 0.5, 10.0))
    return logits_np / T


def softmax_np(logits_np: np.ndarray) -> np.ndarray:
    return torch.softmax(torch.tensor(logits_np), dim=-1).numpy()


# ============================================================================
# Ensemble methods
# ============================================================================

def consensus_majority(p_ref_dict: Dict[str, np.ndarray],
                       thr_dict: Dict[str, float]) -> np.ndarray:
    """Majority vote across thresholded per-model decisions."""
    names = list(thr_dict.keys())
    votes = np.stack(
        [(p_ref_dict[n] >= thr_dict[n]).astype(int) for n in names], axis=1
    )
    return (votes.sum(axis=1) >= math.ceil(votes.shape[1] / 2)).astype(int)


def consensus_unanimous(p_ref_dict: Dict[str, np.ndarray],
                        thr_dict: Dict[str, float]) -> np.ndarray:
    names = list(thr_dict.keys())
    votes = np.stack(
        [(p_ref_dict[n] >= thr_dict[n]).astype(int) for n in names], axis=1
    )
    return (votes.sum(axis=1) == votes.shape[1]).astype(int)


def weighted_ensemble_score(score_dict: Dict[str, np.ndarray],
                            weights: Dict[str, float]) -> np.ndarray:
    names = list(score_dict.keys())
    w = np.array([weights.get(n, 1.0) for n in names], dtype=np.float32)
    w = w / w.sum()
    stacked = np.stack([score_dict[n] for n in names], axis=1)
    return (stacked * w.reshape(1, -1)).sum(axis=1)


def fit_stacking_meta_model(
    p_ref_cal: Dict[str, np.ndarray],
    y_cal_bin: np.ndarray,
    model_names: List[str],
    n_folds: int = STACKING_CV_FOLDS,
) -> Dict[str, Any]:
    """Train a logistic-regression meta-learner on CAL via stratified CV.

    Returns out-of-fold predictions (for diagnostic AUC and threshold
    selection) plus a final model fit on all of CAL.
    """
    missing = [m for m in model_names if m not in p_ref_cal]
    if missing:
        raise ValueError(f"Missing scores for: {missing}")

    X = np.stack([p_ref_cal[n] for n in model_names], axis=1)
    y = np.asarray(y_cal_bin).astype(int)

    # Make sure we have enough samples per class for the chosen fold count.
    folds = n_folds
    if len(y) < folds:
        folds = max(2, len(y))
    if len(np.unique(y)) > 1:
        folds = max(2, min(folds, int(np.min(np.bincount(y)))))

    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y), dtype=float)
    for tr_idx, va_idx in skf.split(X, y):
        clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
        clf.fit(X[tr_idx], y[tr_idx])
        oof[va_idx] = clf.predict_proba(X[va_idx])[:, 1]

    final_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    final_model.fit(X, y)

    return {
        "oof_preds_cal": oof,
        "final_model": final_model,
        "model_names": list(model_names),
        "n_folds_used": int(folds),
    }


def stacking_predict(meta: Dict[str, Any], p_ref: Dict[str, np.ndarray]) -> np.ndarray:
    X = np.stack([p_ref[n] for n in meta["model_names"]], axis=1)
    return meta["final_model"].predict_proba(X)[:, 1]


# ============================================================================
# Two-stage referral with agreement-based veto
# ============================================================================

def apply_agreement_veto(
    stage1_pred: np.ndarray,
    p_ref: Dict[str, np.ndarray],
    veto_models: List[str],
    veto_thr: float,
    min_agree: int,
) -> np.ndarray:
    """Flip a stage-1 positive to negative when at least `min_agree`
    refinement models score it below `veto_thr`.
    """
    stage1_pred = np.asarray(stage1_pred).astype(int)
    low_votes = np.zeros_like(stage1_pred, dtype=int)
    for m in veto_models:
        low_votes += (np.asarray(p_ref[m]) < float(veto_thr)).astype(int)

    veto_triggered = low_votes >= int(min_agree)
    out = stage1_pred.copy()
    out[(stage1_pred == 1) & veto_triggered] = 0
    return out


def tune_veto_threshold(
    y_true_bin: np.ndarray,
    stage1_pred: np.ndarray,
    p_ref: Dict[str, np.ndarray],
    veto_models: List[str],
    target_sens: float,
    grid: np.ndarray = VETO_THR_GRID,
    min_agree: int = VETO_MIN_AGREE,
) -> Dict[str, Any]:
    """Pick the veto threshold that minimises referrals while keeping
    sensitivity at or above `target_sens`. If no threshold qualifies the
    veto is left disabled.
    """
    y_true_bin = np.asarray(y_true_bin).astype(int)
    stage1_pred = np.asarray(stage1_pred).astype(int)

    base = referral_stats(y_true_bin, stage1_pred, "stage1")
    if base["sensitivity"] < target_sens:
        return {
            "veto_thr": None,
            "min_agree": int(min_agree),
            "veto_disabled_reason": "stage1_sensitivity_below_target",
            **base,
        }

    best: Optional[Dict[str, Any]] = None
    for vt in sorted(grid, reverse=True):
        y_hat = apply_agreement_veto(stage1_pred, p_ref, veto_models,
                                     veto_thr=float(vt), min_agree=min_agree)
        rep = referral_stats(y_true_bin, y_hat, "candidate")
        if rep["sensitivity"] < target_sens:
            continue

        if (best is None
                or rep["referrals"] < best["referrals"]
                or (rep["referrals"] == best["referrals"]
                    and rep["specificity"] > best["specificity"])):
            best = {"veto_thr": float(vt), "min_agree": int(min_agree), **rep}

    if best is None:
        return {
            "veto_thr": None,
            "min_agree": int(min_agree),
            "veto_disabled_reason": "no_threshold_meets_sensitivity",
            **base,
        }

    final = apply_agreement_veto(stage1_pred, p_ref, veto_models,
                                 veto_thr=best["veto_thr"], min_agree=min_agree)
    ok, _ = sensitivity_is_met(y_true_bin, final, target_sens)
    if not ok:
        return {
            "veto_thr": None,
            "min_agree": int(min_agree),
            "veto_disabled_reason": "verification_failed",
            **base,
        }
    return best


# ============================================================================
# Reporting helpers
# ============================================================================

def referral_stats(y_true: np.ndarray, y_pred: np.ndarray, name: str) -> Dict[str, Any]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    total = len(y_true)
    referrals = fp + tp
    return {
        "model": name,
        "total_cases": int(total),
        "referrals": int(referrals),
        "true_positives": int(tp),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_negatives": int(tn),
        "sensitivity": float(tp / max(1, tp + fn)),
        "specificity": float(tn / max(1, tn + fp)),
        "waste_rate": float(fp / max(1, referrals)),
        "referral_rate": float(referrals / max(1, total)),
    }


def cost_columns(row: pd.Series, cost_per_ref: float = COST_PER_REFERRAL,
                 screenings: int = SCREENINGS_BASE) -> pd.Series:
    total = max(1, row["total_cases"])
    waste_per_1k = (row["false_positives"] / total) * screenings
    return pd.Series({
        "referrals_per_1000": float((row["referrals"] / total) * screenings),
        "wasted_referrals_per_1000": float(waste_per_1k),
        "wasted_cost_per_1000_usd": float(waste_per_1k * cost_per_ref),
    })


def save_confusion_plot(y_true, y_pred, labels, title: str, out_png: Path) -> np.ndarray:
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(5.8, 5.0))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(
        ax=ax, colorbar=True, values_format="d"
    )
    ax.set_title(title)
    plt.tight_layout()
    fig.savefig(out_png, dpi=180)
    plt.close(fig)
    return cm


def save_roc_plot(y_true_bin, score, title: str, out_png: Path) -> float:
    fpr, tpr, _ = roc_curve(y_true_bin, score)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6.0, 4.8))
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend(loc="lower right")
    plt.tight_layout()
    fig.savefig(out_png, dpi=180)
    plt.close(fig)
    return float(roc_auc)


# ============================================================================
# Second-stage reversal audit
# ============================================================================
#
# A "reversed" referral is a case marked positive by stage 1 and negative by
# the final stage-2 pipeline. Stage 1 is intentionally aggressive: it will
# refer borderline cases. Stage 2 then checks whether the rest of the
# ensemble agrees. When too many refinement models disagree, the referral is
# dropped. The audit records the numerical reason for each reversal in both
# CSV and plain-text form.
#
# Important caveat: these explanations describe the *model's* decision logic.
# They do not constitute clinical proof that pathology is absent.

def _classify_reversal_reason(low_vote_count: int, min_agree: int,
                              stage1_score: float, stage1_thr: float,
                              near_margin: float = 0.05) -> str:
    margin = float(stage1_score - stage1_thr)
    if low_vote_count >= min_agree:
        if margin < near_margin:
            return "Borderline stage-1 positive overturned by stage-2 disagreement"
        return "Stage-1 positive overturned by ensemble veto"
    return "Secondary refinement override"


def build_reversal_audit(
    df_test: pd.DataFrame,
    p_ref_test: Dict[str, np.ndarray],
    stage1_pred_test: np.ndarray,
    final_preds_by_mode: Dict[str, np.ndarray],
    deploy_plan: Dict[str, Dict[str, Any]],
    consensus_models: List[str],
    stage1_model: str,
) -> Dict[str, pd.DataFrame]:
    """For each operating mode, build a row for every case where stage 1
    was positive and the final pipeline was negative.
    """
    audits: Dict[str, pd.DataFrame] = {}

    for mode_key, plan in deploy_plan.items():
        veto_thr = plan["veto_thr_cal"]
        min_agree = plan["veto_min_agree"]
        target_sens = plan["target_sens"]
        stage1_thr = float(plan["stage1_thr_cal"])

        stage1 = np.asarray(stage1_pred_test).astype(int)
        final = np.asarray(final_preds_by_mode[mode_key]).astype(int)
        reversed_idx = np.where((stage1 == 1) & (final == 0))[0]

        rows: List[Dict[str, Any]] = []
        for i in reversed_idx:
            row = df_test.iloc[i]
            stage1_score = float(p_ref_test[stage1_model][i])
            margin = stage1_score - stage1_thr
            model_scores = {m: float(p_ref_test[m][i]) for m in consensus_models}

            if veto_thr is not None:
                low = [m for m in consensus_models if model_scores[m] < float(veto_thr)]
                high = [m for m in consensus_models if model_scores[m] >= float(veto_thr)]
                low_n = len(low)
                triggered = low_n >= int(min_agree)
            else:
                low, high, low_n, triggered = [], list(consensus_models), 0, False

            sorted_scores = sorted(model_scores.items(), key=lambda kv: kv[1])
            category = _classify_reversal_reason(low_n, min_agree, stage1_score, stage1_thr)

            if veto_thr is None:
                reason = (
                    f"Stage 1 marked this case positive ({stage1_model} score={stage1_score:.4f} "
                    f">= threshold={stage1_thr:.4f}). Mode '{mode_key}' has no active veto "
                    f"threshold, so a reversal is unexpected and should be reviewed."
                )
                inference = (
                    "This case was referred at stage 1 but somehow filtered later despite no "
                    "active veto. Likely a pipeline-consistency issue to inspect manually."
                )
            else:
                reason = (
                    f"Stage 1 marked this case positive ({stage1_model} score={stage1_score:.4f} "
                    f">= threshold={stage1_thr:.4f}). Stage 2 reversed it because "
                    f"{low_n}/{len(consensus_models)} refinement models had referral probability "
                    f"below the veto threshold {float(veto_thr):.4f}, meeting the minimum "
                    f"low-agreement count of {int(min_agree)}. "
                    f"Low-support: {', '.join(low) if low else 'none'}."
                )
                if margin < 0.05:
                    inference = (
                        f"Borderline stage-1 referral: the case barely exceeded the safety "
                        f"threshold (margin={margin:.4f}) and the ensemble did not provide "
                        f"enough additional support to keep it."
                    )
                else:
                    inference = (
                        "A stronger stage-1 candidate that was overturned because multiple "
                        "refinement models disagreed. The ensemble interpreted the evidence as "
                        "insufficient for referral."
                    )

            entry: Dict[str, Any] = {
                "test_index": int(i),
                "image_path": str(row["proc_path"]),
                "patient_id": str(row.get("patient_id", "")),
                "true_level_5class": int(row["level"]),
                "true_binary": int(int(row["level"]) >= 1),

                "mode": mode_key,
                "mode_label": plan["mode_label"],
                "target_sensitivity": float(target_sens),

                "stage1_model": stage1_model,
                "stage1_score": stage1_score,
                "stage1_threshold": stage1_thr,
                "stage1_margin_above_threshold": float(margin),
                "stage1_positive": int(stage1[i]),

                "final_prediction": int(final[i]),
                "veto_threshold": None if veto_thr is None else float(veto_thr),
                "veto_min_agree": int(min_agree),
                "low_vote_count": int(low_n),
                "veto_triggered": bool(triggered),

                "low_support_models": ", ".join(low),
                "high_support_models": ", ".join(high),
                "sorted_model_scores": " | ".join(f"{m}:{s:.4f}" for m, s in sorted_scores),

                "reason_category": category,
                "reason_text": reason,
                "inference_text": inference,
            }
            entry.update({f"{m}_score": float(model_scores[m]) for m in consensus_models})
            rows.append(entry)

        audits[mode_key] = pd.DataFrame(rows)

    return audits


def save_audit_text_reports(audits: Dict[str, pd.DataFrame], out_dir: Path) -> List[str]:
    """Write one plain-text explanation per reversed case."""
    out_dir.mkdir(parents=True, exist_ok=True)
    saved: List[str] = []

    for mode_key, df_audit in audits.items():
        mode_dir = out_dir / mode_key
        mode_dir.mkdir(parents=True, exist_ok=True)

        if df_audit.empty:
            note = mode_dir / "no_reversals_found.txt"
            note.write_text("No stage-1 positive to stage-2 negative cases found.\n", encoding="utf-8")
            saved.append(str(note))
            continue

        for _, r in df_audit.iterrows():
            base = Path(r["image_path"]).stem
            path = mode_dir / f"{int(r['test_index']):04d}_{base}_reversal.txt"
            path.write_text(
                "SECOND-STAGE REVERSAL EXPLANATION\n"
                "=================================\n"
                f"Mode: {r['mode']} ({r['mode_label']})\n"
                f"Patient ID: {r['patient_id']}\n"
                f"Image: {r['image_path']}\n"
                f"True 5-class level: {r['true_level_5class']}\n"
                f"True binary label: {r['true_binary']}\n\n"
                "Stage 1\n"
                "-------\n"
                f"Model: {r['stage1_model']}\n"
                f"Score: {r['stage1_score']:.4f}\n"
                f"Threshold: {r['stage1_threshold']:.4f}\n"
                f"Margin above threshold: {r['stage1_margin_above_threshold']:.4f}\n\n"
                "Stage 2\n"
                "-------\n"
                f"Final prediction: {r['final_prediction']}\n"
                f"Veto threshold: {r['veto_threshold']}\n"
                f"Minimum low-agreement required: {r['veto_min_agree']}\n"
                f"Low vote count: {r['low_vote_count']}\n"
                f"Veto triggered: {r['veto_triggered']}\n\n"
                "Model scores\n"
                "------------\n"
                f"{r['sorted_model_scores']}\n\n"
                f"Low-support models: {r['low_support_models']}\n"
                f"High-support models: {r['high_support_models']}\n\n"
                f"Reason category: {r['reason_category']}\n\n"
                "Pipeline reasoning\n"
                "------------------\n"
                f"{r['reason_text']}\n\n"
                "High-level inference\n"
                "--------------------\n"
                f"{r['inference_text']}\n",
                encoding="utf-8",
            )
            saved.append(str(path))

    return saved


def summarise_reversals(audits: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    summary: List[Dict[str, Any]] = []
    for mode_key, df_audit in audits.items():
        if df_audit.empty:
            summary.append({
                "mode": mode_key,
                "num_reversals": 0,
                "avg_stage1_score": float("nan"),
                "avg_margin_above_threshold": float("nan"),
                "avg_low_vote_count": float("nan"),
                "most_common_reason": "None",
                "most_common_low_support_pattern": "None",
            })
            continue

        cats = df_audit["reason_category"].value_counts()
        patterns = df_audit["low_support_models"].value_counts()
        summary.append({
            "mode": mode_key,
            "num_reversals": int(len(df_audit)),
            "avg_stage1_score": float(df_audit["stage1_score"].mean()),
            "avg_margin_above_threshold": float(df_audit["stage1_margin_above_threshold"].mean()),
            "avg_low_vote_count": float(df_audit["low_vote_count"].mean()),
            "most_common_reason": cats.index[0] if len(cats) else "None",
            "most_common_low_support_pattern": patterns.index[0] if len(patterns) else "None",
        })

    return pd.DataFrame(summary)


def save_audit_plots(audits: Dict[str, pd.DataFrame], out_dir: Path) -> List[str]:
    out_dir.mkdir(parents=True, exist_ok=True)
    saved: List[str] = []
    for mode_key, df_audit in audits.items():
        if df_audit.empty:
            continue

        # Stage-1 margin histogram
        fig, ax = plt.subplots(figsize=(8, 4.8))
        ax.hist(df_audit["stage1_margin_above_threshold"], bins=15)
        ax.set_xlabel("Stage-1 margin above threshold")
        ax.set_ylabel("Count")
        ax.set_title(f"Reversed referrals — stage-1 margin ({mode_key})")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        p1 = out_dir / f"{mode_key}_reversal_margin_hist.png"
        fig.savefig(p1, dpi=180)
        plt.close(fig)
        saved.append(str(p1))

        # Low-vote bar
        fig, ax = plt.subplots(figsize=(6.8, 4.8))
        df_audit["low_vote_count"].value_counts().sort_index().plot(kind="bar", ax=ax)
        ax.set_xlabel("Number of low-support refinement models")
        ax.set_ylabel("Count")
        ax.set_title(f"Reversed referrals — low-support votes ({mode_key})")
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        p2 = out_dir / f"{mode_key}_reversal_low_vote_bar.png"
        fig.savefig(p2, dpi=180)
        plt.close(fig)
        saved.append(str(p2))

    return saved


# ============================================================================
# Grad-CAM and per-case explainability
# ============================================================================

_IMAGENET_MEAN_NP = np.array(IMAGENET_MEAN, dtype=np.float32)
_IMAGENET_STD_NP = np.array(IMAGENET_STD, dtype=np.float32)


def load_image_for_inference(image_path: str, size: int,
                             device: torch.device) -> Tuple[Image.Image, Image.Image, torch.Tensor]:
    img = Image.open(image_path)
    if img.mode != "RGB":
        img = img.convert("RGB")
    img_resized = img.resize((size, size), Image.BILINEAR)
    arr = np.asarray(img_resized).astype(np.float32) / 255.0
    arr = (arr - _IMAGENET_MEAN_NP) / _IMAGENET_STD_NP
    tensor = torch.tensor(np.transpose(arr, (2, 0, 1)), dtype=torch.float32).unsqueeze(0).to(device)
    return img, img_resized, tensor


def denormalise_for_display(x_chw: torch.Tensor) -> np.ndarray:
    arr = x_chw.detach().cpu().numpy().transpose(1, 2, 0)
    return np.clip(arr * _IMAGENET_STD_NP + _IMAGENET_MEAN_NP, 0, 1)


def heatmap_overlay(image_rgb: np.ndarray, cam: np.ndarray, alpha: float = 0.38) -> np.ndarray:
    cmap = plt.get_cmap("jet")
    heat = cmap(cam)[..., :3]
    return np.clip((1 - alpha) * image_rgb + alpha * heat, 0, 1)


def find_resnet_target_layer(model: nn.Module) -> nn.Module:
    """For Grad-CAM, we want the last conv block before the classifier head."""
    if hasattr(model, "layer4") and len(model.layer4) > 0:
        return model.layer4[-1]
    last_conv: Optional[nn.Module] = None
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            last_conv = m
    if last_conv is None:
        raise RuntimeError("No Conv2d layer found for Grad-CAM.")
    return last_conv


class GradCAM:
    """Minimal Grad-CAM implementation.

    Register forward and full backward hooks on `target_layer`, then call
    the instance with an input tensor and a target class index.
    """

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model.eval()
        self.target_layer = target_layer
        self.activations: Optional[torch.Tensor] = None
        self.gradients: Optional[torch.Tensor] = None

        self._fwd = target_layer.register_forward_hook(self._save_activations)
        self._bwd = target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, _module, _inp, out):
        self.activations = out.detach()

    def _save_gradients(self, _module, _grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def close(self) -> None:
        self._fwd.remove()
        self._bwd.remove()

    def __call__(self, x: torch.Tensor, target_class: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        if logits.ndim != 2 or logits.shape[0] != 1:
            raise RuntimeError(f"Expected (1, num_classes) logits, got {tuple(logits.shape)}")

        probs = F.softmax(logits, dim=1)
        logits[0, target_class].backward(retain_graph=True)

        acts = self.activations[0]      # type: ignore[index]
        grads = self.gradients[0]       # type: ignore[index]
        weights = grads.mean(dim=(1, 2), keepdim=True)
        cam = F.relu((weights * acts).sum(dim=0))

        if float(cam.max()) > 0:
            cam = cam / (cam.max() + 1e-8)

        cam = cam.unsqueeze(0).unsqueeze(0)
        cam = F.interpolate(cam, size=(x.shape[2], x.shape[3]),
                            mode="bilinear", align_corners=False)
        cam_np = cam[0, 0].detach().cpu().numpy()
        return cam_np, probs[0].detach().cpu().numpy(), logits[0].detach().cpu().numpy()


def calibrated_referral_score(logits_5: np.ndarray, T: float) -> Tuple[float, np.ndarray]:
    """Apply temperature scaling, then collapse to P(referrable)."""
    logits_T = logits_5 / float(T)
    probs = torch.softmax(torch.tensor(logits_T, dtype=torch.float32), dim=0).numpy()
    return float(1.0 - probs[0]), probs


def _choose_cam_target_class(probs_5: np.ndarray, positive: bool) -> int:
    """For a 'positive' explanation, target the strongest non-class-0 class;
    for a 'negative' explanation, target class 0.
    """
    probs_5 = np.asarray(probs_5)
    if positive:
        return int(np.argmax(probs_5[1:]) + 1)
    return 0


def _build_explanation_text(stage1_score: float, stage1_thr: Optional[float],
                            stage1_pred: int, final_pred: int,
                            per_model_scores: Dict[str, float],
                            veto_thr: Optional[float], min_agree: Optional[int]) -> str:
    lines: List[str] = []
    if stage1_thr is not None:
        lines.append(f"Stage 1 score={stage1_score:.4f} vs threshold={stage1_thr:.4f}.")
    else:
        lines.append(f"Stage 1 score={stage1_score:.4f}.")

    lines.append(
        "Stage 1 considered the image referrable."
        if stage1_pred == 1
        else "Stage 1 considered the image non-referrable."
    )

    if veto_thr is not None and min_agree is not None:
        low = [k for k, v in per_model_scores.items() if v < float(veto_thr)]
        high = [k for k, v in per_model_scores.items() if v >= float(veto_thr)]
        lines.append(
            f"Stage 2 used veto threshold={veto_thr:.4f} with min low-agreement={int(min_agree)}."
        )
        lines.append(f"Low-support ({len(low)}): {', '.join(low) if low else 'none'}.")
        lines.append(f"High-support ({len(high)}): {', '.join(high) if high else 'none'}.")

        if stage1_pred == 1 and final_pred == 0:
            lines.append(
                "Stage 1 referral overturned because too many refinement models scored low."
            )
        elif stage1_pred == 1 and final_pred == 1:
            lines.append("Referral retained: ensemble support was strong enough.")
        elif stage1_pred == 0 and final_pred == 0:
            lines.append("Case stayed negative throughout the pipeline.")
        else:
            lines.append("Final decision differs from stage 1 after ensemble refinement.")
    else:
        lines.append(
            "Final output: referred." if final_pred == 1 else "Final output: not referred."
        )

    if final_pred == 1:
        lines.append("Highlighted regions are the areas that most increased referral evidence.")
    else:
        lines.append("The model did not accumulate strong referral evidence here.")

    return " ".join(lines)


def explain_single_image(
    image_path: str,
    true_level: Optional[int],
    models_dict: Dict[str, nn.Module],
    temps: Dict[str, float],
    stage1_model_name: str,
    stage1_thr: Optional[float],
    consensus_models: List[str],
    veto_thr: Optional[float],
    min_agree: int,
    gradcam: GradCAM,
    device: torch.device,
    out_dir: Path,
    save_prefix: Optional[str] = None,
) -> Dict[str, Any]:
    """Generate Grad-CAM overlays and a per-case report for one image."""
    _, _, x = load_image_for_inference(image_path, IMAGE_SIZE, device)

    # First pass: get calibrated stage-1 probs to pick CAM target classes.
    with torch.no_grad():
        stage1_logits = models_dict[stage1_model_name](x)[0].detach().cpu().numpy()
    stage1_score, probs_stage1 = calibrated_referral_score(stage1_logits, temps[stage1_model_name])

    pos_target = _choose_cam_target_class(probs_stage1, positive=True)
    neg_target = _choose_cam_target_class(probs_stage1, positive=False)

    # Grad-CAM for both targets (needs grad enabled)
    with torch.enable_grad():
        cam_pos, _, _ = gradcam(x, target_class=pos_target)
        cam_neg, _, _ = gradcam(x, target_class=neg_target)

    stage1_pred = int(stage1_score >= (stage1_thr if stage1_thr is not None else 0.5))

    # Calibrated p_ref for every model (needed for veto)
    per_model_scores: Dict[str, float] = {}
    per_model_probs5: Dict[str, List[float]] = {}
    for name, model in models_dict.items():
        model.eval()
        with torch.no_grad():
            logits = model(x)[0].detach().cpu().numpy()
        score, probs5 = calibrated_referral_score(logits, temps[name])
        per_model_scores[name] = float(score)
        per_model_probs5[name] = probs5.tolist()

    final_pred = stage1_pred
    if veto_thr is not None:
        low_support = sum(1 for m in consensus_models if per_model_scores[m] < float(veto_thr))
        if stage1_pred == 1 and low_support >= int(min_agree):
            final_pred = 0

    # ---- Plot ----
    x_disp = denormalise_for_display(x[0])
    overlay_pos = heatmap_overlay(x_disp, cam_pos, alpha=0.40)
    overlay_neg = heatmap_overlay(x_disp, cam_neg, alpha=0.40)

    stem = Path(image_path).stem
    prefix = save_prefix or stem
    out_png = out_dir / f"{prefix}_gradcam_report.png"
    out_json = out_dir / f"{prefix}_explanation.json"
    out_txt = out_dir / f"{prefix}_explanation.txt"

    fig = plt.figure(figsize=(15, 9))
    panels = [
        (1, x_disp, "Input image"),
        (2, cam_pos, "Grad-CAM: referral evidence"),
        (3, overlay_pos, "Overlay: referral evidence"),
        (4, x_disp, "Input image"),
        (5, cam_neg, "Grad-CAM: class-0 evidence"),
        (6, overlay_neg, "Overlay: class-0 evidence"),
    ]
    for idx, data, title in panels:
        ax = plt.subplot(2, 3, idx)
        cmap = "jet" if data.ndim == 2 else None
        ax.imshow(data, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")

    title_lines = [
        f"Pipeline explainability — {stem}",
        f"Stage1 score={stage1_score:.4f}  |  Stage1 pred={stage1_pred}  |  Final pred={final_pred}",
    ]
    if true_level is not None:
        title_lines.append(
            f"True level={int(true_level)}  |  True binary={int(int(true_level) >= 1)}"
        )
    plt.suptitle("\n".join(title_lines), fontsize=14, y=0.98)
    plt.tight_layout(rect=[0, 0.04, 1, 0.92])
    fig.savefig(out_png, dpi=180, bbox_inches="tight")
    plt.close(fig)

    reason = _build_explanation_text(
        stage1_score=stage1_score,
        stage1_thr=stage1_thr,
        stage1_pred=stage1_pred,
        final_pred=final_pred,
        per_model_scores=per_model_scores,
        veto_thr=veto_thr,
        min_agree=min_agree,
    )

    report = {
        "image_path": str(image_path),
        "image_stem": stem,
        "true_level": None if true_level is None else int(true_level),
        "true_binary": None if true_level is None else int(int(true_level) >= 1),
        "stage1_model": stage1_model_name,
        "stage1_score": float(stage1_score),
        "stage1_threshold": None if stage1_thr is None else float(stage1_thr),
        "stage1_prediction": int(stage1_pred),
        "veto_threshold": None if veto_thr is None else float(veto_thr),
        "min_low_agreement_required": int(min_agree),
        "final_prediction": int(final_pred),
        "per_model_scores": per_model_scores,
        "per_model_probs5": per_model_probs5,
        "positive_cam_target_class": int(pos_target),
        "negative_cam_target_class": int(neg_target),
        "reason_text": reason,
        "report_png": str(out_png),
    }

    out_json.write_text(json.dumps(report, indent=2), encoding="utf-8")

    txt = [
        "PIPELINE EXPLAINABILITY REPORT",
        "=" * 60,
        f"Image: {image_path}",
    ]
    if true_level is not None:
        txt += [
            f"True 5-class level: {int(true_level)}",
            f"True binary label: {int(int(true_level) >= 1)}",
        ]
    txt += [
        "",
        f"Stage 1 model: {stage1_model_name}",
        f"Stage 1 score: {stage1_score:.4f}",
    ]
    if stage1_thr is not None:
        txt.append(f"Stage 1 threshold: {stage1_thr:.4f}")
    txt.append(f"Stage 1 prediction: {stage1_pred}")
    if veto_thr is not None:
        txt += [
            f"Veto threshold: {veto_thr:.4f}",
            f"Minimum low-agreement required: {int(min_agree)}",
        ]
    txt += [
        f"Final prediction: {final_pred}",
        "",
        "Model scores",
        "-" * 20,
    ]
    txt += [f"{k}: {v:.4f}" for k, v in per_model_scores.items()]
    txt += ["", "Reasoning", "-" * 20, reason]
    out_txt.write_text("\n".join(txt) + "\n", encoding="utf-8")

    return report


# ============================================================================
# Output bookkeeping
# ============================================================================

def copy_artifacts_to_public(files_dir: Path, sources: List[Path]) -> List[str]:
    files_dir.mkdir(parents=True, exist_ok=True)
    copied: List[str] = []
    for p in sources:
        p = Path(p)
        if not p.exists():
            continue
        dst = files_dir / p.name
        shutil.copy2(p, dst)
        copied.append(str(dst))
    return copied


def add_dir_to_zip(zf: zipfile.ZipFile, folder: Path, arc_prefix: str) -> None:
    folder = Path(folder)
    if not folder.exists():
        return
    for fp in folder.rglob("*"):
        if fp.is_file():
            zf.write(fp, arcname=str(Path(arc_prefix) / fp.relative_to(folder)))


def maybe_download_colab(path: Path) -> None:
    """Trigger a Colab download if available. No-op otherwise."""
    try:
        from google.colab import files as colab_files  # type: ignore
        colab_files.download(str(path))
    except Exception:
        pass


# ============================================================================
# Main orchestration
# ============================================================================

def main() -> None:
    # --- setup ---
    set_seed(SEED)
    try_mount_colab_drive()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Pipeline version : {PIPELINE_VERSION}")
    print(f"Device           : {device}")
    print(f"Random seed      : {SEED}")

    for d in [Path(ROOT_OUT), Path(FILES_DIR), Path(MODELS_DIR),
              CAL_DIR, TEST_DIR, AUDIT_DIR, AUDIT_TEXT_DIR, AUDIT_PLOTS_DIR,
              EXPLAIN_DIR, POSTER_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    # --- data ---
    safe_unzip(ZIP_PATH, BASE_UNZIP_DIR)

    images_dir = IMAGES_DIR_CONFIG
    if not directory_has_images(images_dir):
        guess = find_image_directory(BASE_UNZIP_DIR, min_images=1)
        if guess:
            print(f"[data] Auto-detected image directory: {guess}")
            images_dir = guess
    if not directory_has_images(images_dir):
        print("[data] No real images found — falling back to synthetic dataset.")
        images_dir = generate_synthetic_dataset(BASE_UNZIP_DIR, LABELS_CSV,
                                                n_per_class=40, size=300)
    print(f"[data] images_dir = {images_dir}")

    labels = pd.read_csv(LABELS_CSV)
    if not {"image", "level"}.issubset(labels.columns):
        raise SystemExit("LABELS_CSV must have columns: image, level")
    labels["image"] = labels["image"].astype(str)
    labels["level"] = labels["level"].astype(int)
    label_map = dict(zip(labels["image"], labels["level"]))

    matched: List[Dict[str, Any]] = []
    for p in list_images(images_dir):
        lvl = label_map.get(Path(p).stem)
        if lvl is not None:
            matched.append({"filename": p, "level": int(lvl)})
    df_raw = pd.DataFrame(matched)
    if df_raw.empty:
        raise SystemExit("No matched images found between IMAGES_DIR and LABELS_CSV.")
    print(f"[data] Matched {len(df_raw)} images to labels.")

    # --- preprocess to disk ---
    proc_dir = Path(ROOT_OUT) / f"processed_{IMAGE_SIZE}"
    df_all = preprocess_all_images(df_raw, str(proc_dir), IMAGE_SIZE)
    df_all["patient_id"] = df_all["proc_path"].apply(extract_patient_id)

    n_patients = df_all["patient_id"].nunique()
    print(f"[data] {len(df_all)} images from {n_patients} unique patients")
    ipp = df_all.groupby("patient_id").size()
    print(f"[data] images/patient  min={ipp.min()}  max={ipp.max()}  median={ipp.median():.1f}")

    # --- patient-disjoint split ---
    df_train, df_cal, df_test = make_patient_disjoint_split(
        df_all, TEST_FRAC, CAL_FRAC_OF_REMAIN, SEED
    )
    print(f"[split] train: {len(df_train):5d} images, {df_train['patient_id'].nunique()} patients")
    print(f"[split] cal  : {len(df_cal):5d} images, {df_cal['patient_id'].nunique()} patients")
    print(f"[split] test : {len(df_test):5d} images, {df_test['patient_id'].nunique()} patients")
    print(f"[split] train class dist: {dict(Counter(df_train['level']))}")
    print(f"[split] cal   class dist: {dict(Counter(df_cal['level']))}")
    print(f"[split] test  class dist: {dict(Counter(df_test['level']))}")

    df_train_capped = stratified_cap_per_class(df_train, "level", MAX_TRAIN_PER_CLASS, SEED)
    print(f"[split] train capped: {len(df_train_capped)} (from {len(df_train)})")

    split_info = {
        "pipeline_version": PIPELINE_VERSION,
        "total_images": int(len(df_all)),
        "total_patients": int(n_patients),
        "train_images": int(len(df_train)),
        "train_patients": int(df_train["patient_id"].nunique()),
        "train_capped_images": int(len(df_train_capped)),
        "cal_images": int(len(df_cal)),
        "cal_patients": int(df_cal["patient_id"].nunique()),
        "test_images": int(len(df_test)),
        "test_patients": int(df_test["patient_id"].nunique()),
        "test_fraction_patients": float(TEST_FRAC),
        "cal_fraction_of_remaining_patients": float(CAL_FRAC_OF_REMAIN),
        "split_method": "GroupShuffleSplit (two-stage, patient-level)",
        "random_seed": int(SEED),
    }
    (Path(ROOT_OUT) / "split_info.json").write_text(json.dumps(split_info, indent=2))

    # --- loaders ---
    train_loader, cal_loader, test_loader = build_loaders(
        df_train_capped, df_cal, df_test, device
    )

    # --- training ---
    criterion = nn.CrossEntropyLoss(weight=class_weight_tensor(df_train_capped, NUM_CLASSES, device))

    print("\n[train] ResNet50")
    resnet, resnet_best = train_classifier(
        build_resnet50(), "resnet50_finetune", RN_EPOCHS, RN_LR,
        train_loader, cal_loader, criterion, device,
    )
    print("\n[train] ViT-B/16")
    vit, vit_best = train_classifier(
        build_vit_b16(), "vit_b16_finetune", VIT_EPOCHS, VIT_LR,
        train_loader, cal_loader, criterion, device,
    )
    print("\n[train] Supervised encoder (ResNet18)")
    encoder, encoder_best = train_classifier(
        SupervisedEncoder(), "encoder_supervised", ENC_EPOCHS, ENC_LR,
        train_loader, cal_loader, criterion, device,
    )

    trained_models: Dict[str, nn.Module] = {
        "resnet50_finetune": resnet,
        "vit_b16_finetune": vit,
        "encoder_supervised": encoder,
    }
    training_results = {
        "resnet50_finetune": resnet_best,
        "vit_b16_finetune": vit_best,
        "encoder_supervised": encoder_best,
    }

    # ------------------------------------------------------------------
    # Calibration and tuning on CAL only
    # ------------------------------------------------------------------
    print("\n[calibrate] Fitting temperatures and per-model thresholds on CAL")

    temps: Dict[str, float] = {}
    p_ref_cal: Dict[str, np.ndarray] = {}
    cal_rows: List[Dict[str, Any]] = []
    y5_cal_ref: Optional[np.ndarray] = None

    for name, model in trained_models.items():
        y5_cal, logits_cal, _ = collect_predictions(model, cal_loader, device)
        if y5_cal_ref is None:
            y5_cal_ref = y5_cal
        ybin_cal = to_binary(y5_cal)

        T = fit_temperature(logits_cal, y5_cal, device)
        temps[name] = T

        probs5_T = softmax_np(apply_temperature(logits_cal, T))
        p_ref_cal[name] = referral_score_from_probs(probs5_T)

        auc_cal = safe_binary_auc(ybin_cal, p_ref_cal[name])
        thr90 = threshold_for_min_sensitivity(ybin_cal, p_ref_cal[name], 0.90)
        thr95 = threshold_for_min_sensitivity(ybin_cal, p_ref_cal[name], 0.95)

        cal_rows.append({
            "model": name,
            "temp_T": T,
            "auc_cal_bin": auc_cal,
            "thr_sens90_cal": thr90["thr"], "sens@thr90_cal": thr90["sens"], "spec@thr90_cal": thr90["spec"],
            "thr_sens95_cal": thr95["thr"], "sens@thr95_cal": thr95["sens"], "spec@thr95_cal": thr95["spec"],
        })
        print(f"  {name:24s}  AUC={auc_cal:.4f}  T={T:.3f}")
        print(f"    sens>=0.90  thr={thr90['thr']:.4f}  sens={thr90['sens']:.4f}  spec={thr90['spec']:.4f}")
        print(f"    sens>=0.95  thr={thr95['thr']:.4f}  sens={thr95['sens']:.4f}  spec={thr95['spec']:.4f}")

    assert y5_cal_ref is not None
    ybin_cal_ref = to_binary(y5_cal_ref)

    df_cal_summary = pd.DataFrame(cal_rows).sort_values("auc_cal_bin", ascending=False).reset_index(drop=True)
    df_cal_summary.to_csv(Path(ROOT_OUT) / "calibration_summary_thresholds.csv", index=False)

    # Diagnostic CAL ROC plots
    for name in trained_models:
        save_roc_plot(ybin_cal_ref, p_ref_cal[name],
                      f"CAL ROC — {name} (calibrated)", CAL_DIR / f"cal_roc_{name}.png")

    # Per-model threshold dicts for consensus
    thr90_by_model = {r["model"]: float(r["thr_sens90_cal"]) for _, r in df_cal_summary.iterrows()}
    thr95_by_model = {r["model"]: float(r["thr_sens95_cal"]) for _, r in df_cal_summary.iterrows()}
    thr90_cons = {m: thr90_by_model[m] for m in CONSENSUS_MODELS}
    thr95_cons = {m: thr95_by_model[m] for m in CONSENSUS_MODELS}

    # Weighted ensemble + stacking on CAL
    auc_weights_cal = {row["model"]: max(1e-6, float(row["auc_cal_bin"]))
                       for _, row in df_cal_summary.iterrows()}
    p_weighted_cal = weighted_ensemble_score(
        {m: p_ref_cal[m] for m in CONSENSUS_MODELS}, auc_weights_cal
    )

    print("\n[stacking] Training meta-model on CAL (with internal CV)")
    stacking = fit_stacking_meta_model(p_ref_cal, ybin_cal_ref, CONSENSUS_MODELS, STACKING_CV_FOLDS)
    stack_oof_cal = stacking["oof_preds_cal"]
    print(f"[stacking] CAL OOF AUC: {safe_binary_auc(ybin_cal_ref, stack_oof_cal):.4f} "
          f"({stacking['n_folds_used']} folds)")

    # Stage 1 threshold
    t1_result = threshold_for_min_sensitivity(
        ybin_cal_ref, p_ref_cal[STAGE1_MODEL_NAME], STAGE1_SENS_TARGET
    )
    stage1_thr_cal = float(t1_result["thr"])
    stage1_pred_cal = (p_ref_cal[STAGE1_MODEL_NAME] >= stage1_thr_cal).astype(int)
    print(f"\n[stage1] {STAGE1_MODEL_NAME} thr={stage1_thr_cal:.4f} "
          f"sens={t1_result['sens']:.4f} spec={t1_result['spec']:.4f}")

    # Veto threshold per operating mode
    deploy_plan: Dict[str, Dict[str, Any]] = {}
    for mode_key, cfg in OPERATING_MODES.items():
        target = float(cfg["target_sens"])
        best = tune_veto_threshold(
            ybin_cal_ref, stage1_pred_cal, p_ref_cal, CONSENSUS_MODELS,
            target_sens=target, grid=VETO_THR_GRID, min_agree=VETO_MIN_AGREE,
        )
        deploy_plan[mode_key] = {
            "mode_label": cfg["label"],
            "target_sens": target,
            "stage1_model": STAGE1_MODEL_NAME,
            "stage1_thr_cal": stage1_thr_cal,
            "veto_min_agree": int(VETO_MIN_AGREE),
            "veto_thr_cal": best["veto_thr"],
            "veto_disabled_reason": best.get("veto_disabled_reason"),
        }
        if best["veto_thr"] is None:
            print(f"[{mode_key}] veto disabled: {best.get('veto_disabled_reason')}")
        else:
            print(f"[{mode_key}] veto thr={best['veto_thr']:.4f}")

    # Save the tuning plan
    tuning_bundle = {
        "pipeline_version": PIPELINE_VERSION,
        "temps_fit_on_cal": temps,
        "thresholds_fit_on_cal": {
            "per_model": df_cal_summary.to_dict(orient="records"),
            "consensus_models": CONSENSUS_MODELS,
            "thr90_cal": thr90_cons,
            "thr95_cal": thr95_cons,
        },
        "weighted_ensemble": {
            "weights_from_cal_auc": auc_weights_cal,
            "models": CONSENSUS_MODELS,
        },
        "stacking": {
            "model_names": stacking["model_names"],
            "n_folds_used": stacking["n_folds_used"],
            "cal_oof_auc_diagnostic": float(safe_binary_auc(ybin_cal_ref, stack_oof_cal)),
        },
        "two_stage": deploy_plan,
    }
    (Path(ROOT_OUT) / "tuning_plan_cal_only.json").write_text(
        json.dumps(tuning_bundle, indent=2, default=str)
    )

    # ------------------------------------------------------------------
    # Evaluation on TEST (no tuning here)
    # ------------------------------------------------------------------
    print("\n[test] Final evaluation on TEST set")

    p_ref_test: Dict[str, np.ndarray] = {}
    y5_test: Optional[np.ndarray] = None

    for name, model in trained_models.items():
        y5_t, logits_t, _ = collect_predictions(model, test_loader, device)
        if y5_test is None:
            y5_test = y5_t
        probs5_T = softmax_np(apply_temperature(logits_t, temps[name]))
        p_ref_test[name] = referral_score_from_probs(probs5_T)

    assert y5_test is not None
    ybin_test = to_binary(y5_test)

    p_weighted_test = weighted_ensemble_score(
        {m: p_ref_test[m] for m in CONSENSUS_MODELS}, auc_weights_cal
    )
    p_stack_test = stacking_predict(stacking, p_ref_test)

    # Consensus modes
    cons_maj_90 = consensus_majority(p_ref_test, thr90_cons)
    cons_maj_95 = consensus_majority(p_ref_test, thr95_cons)
    cons_un_90 = consensus_unanimous(p_ref_test, thr90_cons)
    cons_un_95 = consensus_unanimous(p_ref_test, thr95_cons)

    # Weighted ensemble thresholds taken from CAL
    w_thr90 = threshold_for_min_sensitivity(ybin_cal_ref, p_weighted_cal, 0.90)
    w_thr95 = threshold_for_min_sensitivity(ybin_cal_ref, p_weighted_cal, 0.95)
    y_w90 = (p_weighted_test >= w_thr90["thr"]).astype(int)
    y_w95 = (p_weighted_test >= w_thr95["thr"]).astype(int)

    # Stacking thresholds taken from CAL OOF
    s_thr90 = threshold_for_min_sensitivity(ybin_cal_ref, stack_oof_cal, 0.90)
    s_thr95 = threshold_for_min_sensitivity(ybin_cal_ref, stack_oof_cal, 0.95)
    y_s90 = (p_stack_test >= s_thr90["thr"]).astype(int)
    y_s95 = (p_stack_test >= s_thr95["thr"]).astype(int)

    # Stage-1 only + two-stage outcomes per mode
    stage1_pred_test = (p_ref_test[STAGE1_MODEL_NAME] >= stage1_thr_cal).astype(int)
    final_preds_by_mode: Dict[str, np.ndarray] = {}
    for mode_key, plan in deploy_plan.items():
        vt = plan["veto_thr_cal"]
        if vt is None:
            final_preds_by_mode[mode_key] = stage1_pred_test
        else:
            final_preds_by_mode[mode_key] = apply_agreement_veto(
                stage1_pred_test, p_ref_test, CONSENSUS_MODELS,
                veto_thr=float(vt), min_agree=VETO_MIN_AGREE,
            )

    # Comparison table
    cmp_rows: List[Dict[str, Any]] = []

    def _add(name: str, y_pred: np.ndarray, score: Optional[np.ndarray] = None) -> None:
        rep = referral_stats(ybin_test, y_pred, name)
        rep["auc_bin_test"] = safe_binary_auc(ybin_test, score) if score is not None else float("nan")
        cmp_rows.append(rep)

    for m in trained_models:
        y90 = (p_ref_test[m] >= thr90_by_model[m]).astype(int)
        y95 = (p_ref_test[m] >= thr95_by_model[m]).astype(int)
        _add(f"{m}_thr90", y90, p_ref_test[m])
        _add(f"{m}_thr95", y95, p_ref_test[m])

    _add("consensus_majority_thr90", cons_maj_90, p_weighted_test)
    _add("consensus_majority_thr95", cons_maj_95, p_weighted_test)
    _add("consensus_unanimous_thr90", cons_un_90, p_weighted_test)
    _add("consensus_unanimous_thr95", cons_un_95, p_weighted_test)
    _add("weighted_ensemble_thr90", y_w90, p_weighted_test)
    _add("weighted_ensemble_thr95", y_w95, p_weighted_test)
    _add("stacking_thr90", y_s90, p_stack_test)
    _add("stacking_thr95", y_s95, p_stack_test)
    _add("stage1_only_thr95", stage1_pred_test, p_ref_test[STAGE1_MODEL_NAME])
    for mode_key in OPERATING_MODES:
        _add(f"{mode_key}_two_stage_final", final_preds_by_mode[mode_key], p_weighted_test)

    df_cmp = pd.DataFrame(cmp_rows)
    df_cmp = pd.concat([df_cmp, df_cmp.apply(cost_columns, axis=1)], axis=1)

    baseline_name = "resnet50_finetune_thr95"
    baseline_cost = float(df_cmp.loc[df_cmp["model"] == baseline_name, "wasted_cost_per_1000_usd"].values[0])
    df_cmp["cost_savings_vs_baseline_usd"] = baseline_cost - df_cmp["wasted_cost_per_1000_usd"]
    df_cmp["baseline_model"] = baseline_name
    df_cmp.to_csv(Path(ROOT_OUT) / "FINAL_compare_models_TEST_only.csv", index=False)

    print(f"\n[test] Baseline: {baseline_name}  cost/1000=${baseline_cost:,.2f}")
    print("[test] Top by sensitivity:")
    print(df_cmp.sort_values("sensitivity", ascending=False)[[
        "model", "sensitivity", "specificity", "referrals", "cost_savings_vs_baseline_usd"
    ]].head(12).to_string(index=False))

    # ROC / CM plots on TEST
    for name in trained_models:
        save_roc_plot(ybin_test, p_ref_test[name],
                      f"TEST ROC — {name} (calibrated)", TEST_DIR / f"test_roc_{name}.png")
    save_roc_plot(ybin_test, p_weighted_test, "TEST ROC — Weighted Ensemble",
                  TEST_DIR / "test_roc_weighted_ensemble.png")
    save_roc_plot(ybin_test, p_stack_test, "TEST ROC — Stacking",
                  TEST_DIR / "test_roc_stacking.png")
    save_confusion_plot(ybin_test, stage1_pred_test, [0, 1],
                        "TEST CM — Stage 1 (CAL thr)", TEST_DIR / "test_cm_stage1.png")
    save_confusion_plot(ybin_test, final_preds_by_mode["high_safety"], [0, 1],
                        "TEST CM — Two-stage, High Safety", TEST_DIR / "test_cm_high_safety_final.png")
    save_confusion_plot(ybin_test, final_preds_by_mode["balanced"], [0, 1],
                        "TEST CM — Two-stage, Balanced", TEST_DIR / "test_cm_balanced_final.png")

    # Summary bar charts
    plt.figure(figsize=(12, 5))
    ordered = df_cmp.sort_values("sensitivity", ascending=True)
    plt.bar(ordered["model"], ordered["sensitivity"])
    plt.axhline(y=0.90, linestyle="--", alpha=0.7, label="90% target")
    plt.axhline(y=0.95, linestyle="--", alpha=0.7, label="95% target")
    plt.xticks(rotation=35, ha="right", fontsize=8)
    plt.ylabel("Sensitivity (TEST)")
    plt.title("TEST sensitivity across strategies (CAL-tuned)")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(ROOT_OUT) / "FINAL_TEST_compare_sensitivity.png", dpi=180)
    plt.close()

    plt.figure(figsize=(12, 5))
    ordered = df_cmp.sort_values("false_positives", ascending=True)
    plt.bar(ordered["model"], ordered["false_positives"])
    plt.xticks(rotation=35, ha="right", fontsize=8)
    plt.ylabel("False positives (TEST)")
    plt.title("TEST referral waste — false positives")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(ROOT_OUT) / "FINAL_TEST_compare_false_positives.png", dpi=180)
    plt.close()

    plt.figure(figsize=(12, 5))
    ordered = df_cmp.sort_values("cost_savings_vs_baseline_usd", ascending=True)
    plt.bar(ordered["model"], ordered["cost_savings_vs_baseline_usd"])
    plt.axhline(y=0, color="black", linestyle="-", alpha=0.3)
    plt.xticks(rotation=35, ha="right", fontsize=8)
    plt.ylabel(f"USD saved per 1,000 vs {baseline_name}")
    plt.title("TEST cost savings vs baseline")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(ROOT_OUT) / "FINAL_TEST_compare_cost_savings.png", dpi=180)
    plt.close()

    plt.figure(figsize=(10, 8))
    for _, row in df_cmp.iterrows():
        plt.scatter(row["specificity"] * 100, row["sensitivity"] * 100,
                    s=90, alpha=0.75, edgecolors="black")
    plt.axhline(y=90, linestyle="--", alpha=0.7, label="90% sens")
    plt.axhline(y=95, linestyle="--", alpha=0.7, label="95% sens")
    plt.xlabel("Specificity (%) — TEST")
    plt.ylabel("Sensitivity (%) — TEST")
    plt.title("TEST sensitivity vs specificity trade-off")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(ROOT_OUT) / "FINAL_TEST_sensitivity_vs_specificity.png", dpi=180)
    plt.close()

    # ------------------------------------------------------------------
    # Reversal audit
    # ------------------------------------------------------------------
    print("\n[audit] Building reversal audit (stage-1 positive -> final negative)")
    audits = build_reversal_audit(
        df_test=df_test, p_ref_test=p_ref_test, stage1_pred_test=stage1_pred_test,
        final_preds_by_mode=final_preds_by_mode, deploy_plan=deploy_plan,
        consensus_models=CONSENSUS_MODELS, stage1_model=STAGE1_MODEL_NAME,
    )

    for mode_key, df_audit in audits.items():
        out_csv = AUDIT_DIR / f"reversal_audit_{mode_key}.csv"
        df_audit.to_csv(out_csv, index=False)
        print(f"[audit] {out_csv.name}: {len(df_audit)} reversed cases")

    audit_summary = summarise_reversals(audits)
    audit_summary_csv = AUDIT_DIR / "reversal_audit_summary.csv"
    audit_summary.to_csv(audit_summary_csv, index=False)
    save_audit_text_reports(audits, AUDIT_TEXT_DIR)
    save_audit_plots(audits, AUDIT_PLOTS_DIR)

    narrative = AUDIT_DIR / "reversal_narrative_report.txt"
    with open(narrative, "w", encoding="utf-8") as f:
        f.write("REVERSAL NARRATIVE REPORT\n")
        f.write("=" * 60 + "\n\n")
        f.write(
            "A reversed referral is a case that was positive in stage 1 but "
            "negative after stage 2. Stage 1 is safety-first and may over-call "
            "borderline cases; stage 2 reviews them with the ensemble and "
            "applies an agreement-based veto. If enough refinement models "
            "give weak support, the referral is removed.\n\n"
        )
        f.write("Summary by mode:\n")
        f.write(audit_summary.to_string(index=False))
        f.write("\n\n")
        for mode_key, df_audit in audits.items():
            f.write(f"MODE: {mode_key}\n" + "-" * 60 + "\n")
            if df_audit.empty:
                f.write("No reversed referrals found.\n\n")
                continue
            f.write("Most common reason categories:\n")
            f.write(df_audit["reason_category"].value_counts().to_string() + "\n\n")
            f.write("Most common low-support patterns:\n")
            f.write(df_audit["low_support_models"].value_counts().head(10).to_string() + "\n\n")
            sample = df_audit.iloc[0]
            f.write("Example explanation:\n")
            f.write(sample["reason_text"] + "\n")
            f.write(sample["inference_text"] + "\n\n")

    # ------------------------------------------------------------------
    # Save trained models and run config
    # ------------------------------------------------------------------
    print("\n[save] Persisting model checkpoints and config")

    torch.save({
        "arch": "resnet50",
        "num_classes": NUM_CLASSES,
        "state_dict": resnet.state_dict(),
        "best_cal_auc": training_results["resnet50_finetune"]["cal_auc"],
        "best_epoch": training_results["resnet50_finetune"]["epoch"],
    }, Path(MODELS_DIR) / "resnet50_finetune_5class.pt")

    torch.save({
        "arch": "vit_b_16",
        "num_classes": NUM_CLASSES,
        "state_dict": vit.state_dict(),
        "best_cal_auc": training_results["vit_b16_finetune"]["cal_auc"],
        "best_epoch": training_results["vit_b16_finetune"]["epoch"],
    }, Path(MODELS_DIR) / "vit_b16_finetune_5class.pt")

    torch.save({
        "arch": "supervised_encoder_resnet18",
        "num_classes": NUM_CLASSES,
        "state_dict": encoder.state_dict(),
        "best_cal_auc": training_results["encoder_supervised"]["cal_auc"],
        "best_epoch": training_results["encoder_supervised"]["epoch"],
    }, Path(MODELS_DIR) / "encoder_supervised_5class.pt")

    joblib.dump(stacking["final_model"], Path(MODELS_DIR) / "stacking_meta_model_cal.pkl")

    run_config = {
        "pipeline_version": PIPELINE_VERSION,
        "run_timestamp": datetime.now().isoformat(),
        "seed": SEED,
        "device": str(device),
        "split": split_info,
        "training": {
            "image_size": IMAGE_SIZE,
            "batch_size_train": BATCH_SIZE_TRAIN,
            "batch_size_eval": BATCH_SIZE_EVAL,
            "mixed_precision": USE_MIXED_PRECISION,
            "early_stop_patience": EARLY_STOP_PATIENCE,
            "weight_decay": WEIGHT_DECAY,
            "max_train_per_class": MAX_TRAIN_PER_CLASS,
        },
        "tuning_cal_only": {
            "temps": temps,
            "per_model_thresholds_cal": {
                m: {"thr_sens90": float(thr90_by_model[m]),
                    "thr_sens95": float(thr95_by_model[m])}
                for m in trained_models
            },
            "consensus_models": CONSENSUS_MODELS,
            "weighted_ensemble_weights_from_cal_auc": auc_weights_cal,
            "weighted_ensemble_thr90_cal": float(w_thr90["thr"]),
            "weighted_ensemble_thr95_cal": float(w_thr95["thr"]),
            "stacking_thr90_cal_oof": float(s_thr90["thr"]),
            "stacking_thr95_cal_oof": float(s_thr95["thr"]),
            "stage1_model": STAGE1_MODEL_NAME,
            "stage1_sens_target": STAGE1_SENS_TARGET,
            "stage1_thr_cal": stage1_thr_cal,
            "two_stage_plan": deploy_plan,
        },
        "final_test_reporting": {
            "baseline_model": baseline_name,
            "baseline_cost_per_1000_test": baseline_cost,
            "final_compare_csv": str(Path(ROOT_OUT) / "FINAL_compare_models_TEST_only.csv"),
            "audit_summary_csv": str(audit_summary_csv),
            "audit_narrative_report": str(narrative),
            "note": "All reported metrics are computed on TEST only; no tuning is performed on TEST.",
        },
    }
    (Path(MODELS_DIR) / "run_config.json").write_text(json.dumps(run_config, indent=2, default=str))

    # ------------------------------------------------------------------
    # Grad-CAM explainability for a few representative cases
    # ------------------------------------------------------------------
    print("\n[gradcam] Generating Grad-CAM explanations")

    stage1_resnet = trained_models[STAGE1_MODEL_NAME].eval()
    target_layer = find_resnet_target_layer(stage1_resnet)
    gradcam = GradCAM(stage1_resnet, target_layer)

    # Pick the 'balanced' mode for explanation if it exists, otherwise the first
    explain_mode = "balanced" if "balanced" in OPERATING_MODES else next(iter(OPERATING_MODES))
    explain_plan = deploy_plan[explain_mode]
    explain_veto_thr = explain_plan["veto_thr_cal"]

    # Pick a few representative test cases: top stage-1 positives, top negatives,
    # and a few reversals (if present)
    n_per_bucket = 4
    test_indices_pos = np.argsort(-p_ref_test[STAGE1_MODEL_NAME])[:n_per_bucket]
    test_indices_neg = np.argsort(p_ref_test[STAGE1_MODEL_NAME])[:n_per_bucket]
    final_explain = final_preds_by_mode[explain_mode]
    reversed_idx = np.where((stage1_pred_test == 1) & (final_explain == 0))[0][:n_per_bucket]

    selected = sorted(set(list(test_indices_pos) + list(test_indices_neg) + list(reversed_idx)))
    explain_records: List[Dict[str, Any]] = []
    for i in selected:
        row = df_test.iloc[int(i)]
        try:
            rep = explain_single_image(
                image_path=str(row["proc_path"]),
                true_level=int(row["level"]),
                models_dict=trained_models,
                temps=temps,
                stage1_model_name=STAGE1_MODEL_NAME,
                stage1_thr=stage1_thr_cal,
                consensus_models=CONSENSUS_MODELS,
                veto_thr=explain_veto_thr,
                min_agree=VETO_MIN_AGREE,
                gradcam=gradcam,
                device=device,
                out_dir=EXPLAIN_DIR,
                save_prefix=f"{int(i):04d}_{Path(row['proc_path']).stem}",
            )
            explain_records.append(rep)
        except Exception as exc:
            print(f"[gradcam] Skipping index {i}: {exc}")

    gradcam.close()

    if explain_records:
        pd.DataFrame(explain_records).to_csv(
            EXPLAIN_DIR / "explainability_summary.csv", index=False
        )
        print(f"[gradcam] Wrote {len(explain_records)} reports to {EXPLAIN_DIR}")

    # ------------------------------------------------------------------
    # Copy artifacts to public folder and zip everything
    # ------------------------------------------------------------------
    print("\n[bundle] Copying artifacts to public folder and creating archive")

    sources = [
        Path(ROOT_OUT) / "split_info.json",
        Path(ROOT_OUT) / "tuning_plan_cal_only.json",
        Path(ROOT_OUT) / "calibration_summary_thresholds.csv",
        Path(ROOT_OUT) / "FINAL_compare_models_TEST_only.csv",
        Path(ROOT_OUT) / "FINAL_TEST_compare_sensitivity.png",
        Path(ROOT_OUT) / "FINAL_TEST_compare_false_positives.png",
        Path(ROOT_OUT) / "FINAL_TEST_compare_cost_savings.png",
        Path(ROOT_OUT) / "FINAL_TEST_sensitivity_vs_specificity.png",
        audit_summary_csv,
        narrative,
    ]
    for d in [CAL_DIR, TEST_DIR, AUDIT_DIR, AUDIT_TEXT_DIR, AUDIT_PLOTS_DIR, EXPLAIN_DIR]:
        for p in d.rglob("*"):
            if p.is_file():
                sources.append(p)

    copied = copy_artifacts_to_public(Path(FILES_DIR), sources)
    print(f"[bundle] Copied {len(copied)} files to {FILES_DIR}")

    zip_path = Path(ROOT_OUT) / "DR_pipeline_outputs.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fp in Path(ROOT_OUT).glob("*"):
            if fp.is_file() and fp.name != zip_path.name:
                zf.write(fp, arcname=fp.name)
        add_dir_to_zip(zf, Path(FILES_DIR),    "public")
        add_dir_to_zip(zf, Path(MODELS_DIR),   "models")
        add_dir_to_zip(zf, CAL_DIR,            "calibration_diagnostics")
        add_dir_to_zip(zf, TEST_DIR,           "final_test_results")
        add_dir_to_zip(zf, AUDIT_DIR,          "second_stage_audit")
        add_dir_to_zip(zf, EXPLAIN_DIR,        "explainability_gradcam")

    print(f"[bundle] Archive: {zip_path}")
    maybe_download_colab(zip_path)

    print("\nPipeline complete.")
    print("  train: weights only")
    print("  cal  : temperatures, thresholds, veto, ensemble weights, stacker")
    print("  test : final reporting only")
    print(f"  outputs: {ROOT_OUT}")


if __name__ == "__main__":
    main()